<a href="https://colab.research.google.com/github/BWMIN-Hub/SR_practice/blob/main/notebooks/19_scene_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# One scene, end to end

Pages 13 - 18 took the steps one at a time.  Here they run in order on one THEOS-2
scene, and the run ends with a report that someone else can read without the notebook.

| step | what happens |
|---|---|
| 1  prepare | read the header, set the conversion window |
| 2  infer + write | one block at a time: read, convert, infer in tiles, convert back, write |
| 3  QC | header, radiometry, seams — each check pass or CHECK |
| 4  IQA | no-reference scores against bicubic, and a consistency check against the input |
| 5  report | one HTML file and one JSON with everything above |

The scene is 1400 x 1400 px over Bangkok: four bands, 12-bit in `uint16`, 0.5 m.
The network is an EDSR x2 for THEOS-2.

In [ ]:
import sys, urllib.request

BASE = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main'
for f in ('lib/infer.py', 'lib/iqa.py', 'lib/theos_models.py', 'lib/qcpipe.py'):
    urllib.request.urlretrieve(f'{BASE}/{f}', f.split('/')[-1])
for m in ('infer', 'iqa', 'theos_models', 'qcpipe'):
    sys.modules.pop(m, None)

import os
from IPython.display import HTML
from infer import Config
from qcpipe import *
from theos_models import load_theos_edsr

get = lambda p, out: urllib.request.urlretrieve(f'{BASE}/{p}', out)[0]
PATH = get('dataset/pipeline/scene.tif', 'scene.tif')
WEIGHT = get('models/19_scene_pipeline/checkpoints/edsr_theos2_x2.pt', 'edsr_theos2_x2.pt')
os.makedirs('iqa', exist_ok=True)
for f in ('niqe_model.npz', 'brisque_svr.npz'):
    get(f'results/iqa/{f}', f'iqa/{f}')

## 1. Prepare

The network takes values in 0 - 1.  Each band is stretched linearly from its 2 - 99.5 %
percentile range onto 0 - 1, and the output is mapped back the same way.  After that,
the low frequencies are taken from the input and only the detail from the network.

One window is used for the whole scene, so all blocks come out with the same brightness.

In [ ]:
s = scan(PATH, bands=(1, 2, 3), lo=2, hi=99.5)
show_scan(s)

Near infrared is left out: the network takes three channels.  The output will have
three bands, and the checks below expect three.

## 2. Tiled inference, windowed write

Two levels of cutting.

| level | size | why |
|---|---|---|
| block | 768 px | what is read from disk and written back — bounds the **RAM** |
| tile | 256 px | what goes through the network — bounds the **GPU memory** (page 14) |

Each tile reads 16 px around itself.  Each block is read with the same margin, so block
edges get real neighbours just as tile edges do, and the tiles start at the block
corner, so every tile boundary in the scene lands on a multiple of 256.

The output file is created once, empty, at its final size; each block is written
straight into its window.

In [ ]:
BLOCK = 768                       # a multiple of the tile
cfg = Config(scale=2, tile=256, overlap=16, batch=4, precision='fp32')
net = load_theos_edsr(WEIGHT)
show_blocks(s, BLOCK, cfg)

In [ ]:
st = run(PATH, 'sr.tif', net, cfg, s, block=BLOCK, model='EDSR x2 (THEOS-2)')

The low-frequency step is a second windowed pass over the network output, with its own
margin.  The file records how it was made — model, scale, source and the window are
GeoTIFF tags.

In [ ]:
!gdalinfo sr.tif | grep -E 'Size is|SR_|Band [0-9]'

## 3. QC — header, radiometry, seams

Every check is a row with what was expected and what came out.

| group | asks |
|---|---|
| header | size x2, three bands, same data type and nodata, tiled and compressed, processing tags present |
| radiometry | mean within 2 %, std x 0.9 - 1.3, values in range, histogram shape, no new nodata |
| seams | the step across tile and block boundaries, against the step a few rows away |

All statistics are accumulated strip by strip.

In [ ]:
checks = qc(PATH, 'sr.tif', s, cfg, block=BLOCK)
show_qc(checks)

All 24 pass.

Mean and histogram shape are unchanged — the low frequencies come from the input — and
the standard deviation rose 4 - 6 %: that is the detail the network added.  The step
across a tile or block boundary is 1.05 times the step a few rows away.

The range row passes, but the output stops around 1 700 - 2 300 while the input reaches
3 400 - 4 095: the brightest few pixels lose their peaks.

## 4. IQA

There is no reference image at 0.25 m for this scene, so PSNR and SSIM against a ground
truth are not possible.  Two things are:

- **no-reference scores** — NIQE, BRISQUE, PIQE on SR and on bicubic, lower is better.
  They are calibrated on this project's IKONOS images (page 11), so only the comparison
  between the two means anything.
- **consistency** — shrink the result back to 0.5 m and compare it with the input.  A
  result that changed the scene, rather than only adding detail, does not shrink back
  to what went in.

Three crops are picked automatically: the busiest, a typical and the flattest place.

In [ ]:
res = iqa_scan(PATH, 'sr.tif', s, cfg, iqa_dir='iqa')
checks += iqa_rows(res)
show_iqa(res)

Shrunk back to 0.5 m, the result agrees with the input at 32 - 38 dB (SSIM 0.98 -
0.99): edges are sharper, nothing has moved.  NIQE and BRISQUE are lower than bicubic on
all three crops; PIQE is higher on all three.  Without a reference they can only say
"different", which is why the consistency number is the one that becomes a check.

## 5. The report

One call writes `qc_report.html` — self-contained, every figure embedded — and
`qc_report.json` with the same numbers for a script to read.  The verdict at the top is
PASS only if every check passed.

In [ ]:
html = write_report('qc_report.html', PATH, 'sr.tif', s, st, checks, res)
HTML(html)

The verdict is PASS: all 27 checks.

In Colab the files are in the panel on the left;
`from google.colab import files; files.download('qc_report.html')` saves one.

## 6. When the output is wrong

A copy of the result as it would be if the output had been written as 0 - 255
display values instead of being converted back.  The report only earns its place if it catches this.

In [ ]:
show_qc(qc(PATH, break_output('sr.tif', 'bad.tif', s), s, cfg, block=BLOCK), only_bad=True)

Every radiometry row fails: the mean drops by 87 - 89 %, the values stop at 255, the
histogram shape falls to 0.2, and 290 000 pixels at 0 now read as nodata.  Opened on its
own with an automatic stretch, this file looks fine.